# CE-06 — Failures and Speculative Execution
**Student:** NAGAR SAHAS PRAKASH · Roll No. 202301040048 · Big Data Analytics (CE-06)

Single notebook for the whole practical: **Hadoop 3.4.2** (HDFS + YARN + MapReduce) on WSL2,
11 instrumented job runs covering all 7 required experiments on the NASA Kennedy Space Center HTTP logs.

### How to use this notebook
- Works whether the kernel runs on **Windows** (commands are auto-wrapped with `wsl -d Ubuntu -u root --`) or **inside WSL/Linux**.
- **Evidence cells** pull live data from the JobHistory / ResourceManager REST APIs — they work on the runs already executed (no re-run needed).
- Every experiment section has an optional `RUN = False` cell: flip to `True` to re-run that experiment live (~1–8 min each).
- Evidence folder: `evidence/` · Summary table: `evidence/REPORT_EVIDENCE_SUMMARY.txt` · Full report: `FINAL_REPORT.md`

In [ ]:
# ---------------------------------------------------------------------------
# Setup: paths, REST helpers, shell helper, dependency-free table renderer
# ---------------------------------------------------------------------------
import json, os, shlex, subprocess, urllib.request

PROJ_WIN = 'D:/sem7/BDA-assignment/CE06_Hadoop_Practical'   # forward slashes work in Python on Windows
PROJ_WSL = '/mnt/d/sem7/BDA-assignment/CE06_Hadoop_Practical'
PROJ = PROJ_WSL if os.name == 'posix' else PROJ_WIN

RM  = 'http://localhost:8088'
JHS = 'http://localhost:19888/ws/v1/history/mapreduce/jobs'

def get(url, timeout=20):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return json.load(r)

def sh(cmd, timeout=1500):
    """Run a shell command (auto-wrapped with WSL on Windows)."""
    argv = shlex.split(cmd)
    if os.name == 'nt':
        argv = ['wsl', '-d', 'Ubuntu', '-u', 'root', '--'] + argv
    p = subprocess.run(argv, capture_output=True, text=True, timeout=timeout)
    print((p.stdout + p.stderr).strip()[-4000:])
    return p.returncode

def table(headers, rows):
    """Pretty table (uses pandas if available, plain text otherwise)."""
    if not rows:
        print('(no rows)'); return
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows, columns=headers)); return
    except Exception:
        w = [max([len(str(h))] + [len(str(r[i])) for r in rows]) for i, h in enumerate(headers)]
        sep = ' | '.join('-' * x for x in w)
        print(' | '.join(str(h).ljust(x) for h, x in zip(headers, w)))
        print(sep)
        for r in rows:
            print(' | '.join(str(v).ljust(x) for v, x in zip(r, w)))

print('Project dir :', PROJ)
print('RM API      :', RM)
print('JobHistory  :', JHS)

In [ ]:
# 1) Cluster health check — start the cluster only if the RM is not answering
def rm_up():
    try:
        get(RM + '/ws/v1/cluster/info'); return True
    except Exception:
        return False

print('ResourceManager reachable:', rm_up())
if not rm_up():
    print('Starting cluster...')
    sh('bash ' + PROJ + '/scripts/start_services.sh')

print('\n--- daemons (jps) ---')
sh('jps')

info = get(RM + '/ws/v1/cluster/info')
print('\nHadoop version :', info['clusterInfo']['hadoopVersion'])
print('State          :', info['clusterInfo']['state'])
nodes = get(RM + '/ws/v1/cluster/nodes')['nodes']['node']
table(['Node', 'State', 'Containers', 'UsedMemory'],
      [[n['nodeHostName'] + ':' + n['id'].split(':')[-1], n['state'],
        n['usedContainers'], n['availableMemory'] + '/' + n['memoryCapability'] + ' MB'] for n in nodes])

## 2. Dataset — NASA Kennedy Space Center HTTP logs
Source: https://ita.ee.lbl.gov/traces/ (Jul + Aug 1995). Parsed with the combined-format
tail pattern `"<status> <bytes>"`; non-matching lines are counted and skipped — never silent.

| File | Records | Notes |
|---|---|---|
| `clean/nasa_access_clean.log` (355.8 MB) | 3,461,565 clean | 48 naturally malformed lines skipped |
| `malformed/nasa_access_malformed.log` (359.1 MB) | + 20 injected | marker `CE06_BAD_RECORD`, seed=42, ~every 173k lines |

Injected lines are pure garbage, so any run that *skips* them must produce output
**byte-identical** to the clean baseline — this is used as the correctness proof everywhere.

In [ ]:
# 2) Dataset stats + HDFS listing
with open(os.path.join(PROJ, 'dataset/stats.json')) as f:
    stats = json.load(f)
print(json.dumps(stats, indent=2)[:1500])

print('\n--- HDFS ---')
sh('hdfs dfs -ls -h /data/clean /data/malformed')

print('\n--- injected malformed examples ---')
sh('grep -m 3 CE06_BAD_RECORD ' + PROJ + '/dataset/malformed/nasa_access_malformed.log')

## 3. Application — `FailureSimulation.java`
Mapper extracts the HTTP status code, reducer sums occurrences. Five simulation modes are
selected with `-Dce06.simulation.mode` (all failures are *explicitly thrown/logged* — nothing is silent):

| Mode | Behaviour | Used by |
|---|---|---|
| `NORMAL` | skip + count bad lines | Exp 1 |
| `MALFORMED_RECORD_EXCEPTION` | `IOException` on **every** marker line, every attempt | Exp 2a |
| `FAIL_ONCE` | throw only in attempt `_0`; retry skips the marker | Exp 2b |
| `ALWAYS_FAIL` | throw in `setup()` on partition 0 (input-independent, fails fast) | Exp 3a/3b |
| `SLOW_TASK` | sleep every 1000 lines on partition 0, **attempt `_0` only** (retries/spec run full speed) | Exps 4/5/6/7 |

**Counter group `CE06 Failure Simulation`:** `LINES_READ`, `MALFORMED_MARKER_ENCOUNTERED`,
`BAD_LINE_SKIPPED`, `BAD_LINE_SKIPPED_ON_RETRY`, `UNPARSEABLE_SKIPPED`, `ALWAYS_FAIL_ATTEMPT_THROWN`.

After every job the client prints `CE06-EFFECTIVE-CONF …` so the log itself proves which knobs the job used.

## 4. Cluster configuration (tuned for an 8 GB laptop)
| Setting | Value | Why |
|---|---|---|
| `dfs.replication` | 1 | single DataNode |
| NN/DN/RM/NM/JHS heaps | 256 m | fit WSL's ~3.6 GB memory cap |
| NM resource | 2048 MB / 4 vCores | leave room for daemons |
| map / reduce / AM containers | 512 MB (Xmx400m) | 3 concurrent map slots + AM |
| `maximum-am-resource-percent` | **0.5** (default 0.1) | 0.1x2048=204 MB < 512 MB AM would leave every job stuck `ACCEPTED` |
| split maxsize | 32 MB | 12 map tasks over the 356 MB file |
| NM liveness expiry | 30 s | fast node-failure demos |
| daemons | `setsid --fork` | survive shell/session exit |

In [ ]:
# 4) Effective job configuration, straight from JobHistory REST (proves what the AM actually used)
conf = get(JHS + '/job_1793346852065_0001/conf')['conf']['property']
want = ('mapreduce.map.maxattempts', 'mapreduce.map.speculative', 'mapreduce.reduce.speculative',
        'mapreduce.framework.name', 'mapreduce.input.fileinputformat.split.maxsize',
        'yarn.app.mapreduce.am.resource.mb')
table(['Property', 'Value'], [[p['name'], p['value']] for p in conf if p['name'] in want])

conf5 = get(JHS + '/job_1793346852065_0011/conf')['conf']['property']
spec = [[p['name'], p['value']] for p in conf5 if 'speculative' in p['name'] and 'job' in p['name']]
print('\nSpeculative-related settings of the Exp-5 re-run (job 0011):')
table(['Property', 'Value'], spec)

## 5. Evidence harness
Everything below is pulled live from the JobHistoryServer REST API (the source of truth for
attempt states):
- `GET /ws/v1/history/mapreduce/jobs` → all jobs
- `GET .../jobs/{jobid}/tasks` → task list
- `GET .../jobs/{jobid}/tasks/{taskid}/attempts` → every attempt with **state, duration, diagnostics**
- `GET .../jobs/{jobid}/counters` → MR counters (incl. `Launched map tasks`, `Killed map tasks`)
- `GET http://localhost:8088/ws/v1/cluster/apps/{appid}` → RM record + failure diagnostics

Local mirrors of the same data live in `evidence/job_history/JOB_*_*.json`.

In [ ]:
# 5) Reusable fetchers + overview of ALL runs
LABELS = {
 '0001': ('EXP1  Baseline NORMAL',        'SUCCEEDED (expected)'),
 '0002': ('EXP2a Poison record maxattempts=2', 'FAILED (expected)'),
 '0003': ('EXP2b Fail-once, retry wins',   'SUCCEEDED (expected)'),
 '0004': ('EXP3a ALWAYS_FAIL max=2',       'FAILED (expected)'),
 '0005': ('EXP3b ALWAYS_FAIL max=4',       'FAILED (expected)'),
 '0006': ('EXP4  spec OFF (prelim)',       'SUCCEEDED'),
 '0007': ('EXP5  spec ON  (prelim, no dup)','SUCCEEDED - gotcha'),
 '0008': ('EXP6  kill -9 attempt',         'SUCCEEDED (expected)'),
 '0009': ('EXP7  NodeManager failure',     'SUCCEEDED (expected)'),
 '0010': ('EXP4v2 spec OFF sleep=400',     'SUCCEEDED (final pair)'),
 '0011': ('EXP5v2 spec ON  sleep=400',     'SUCCEEDED (final pair)'),
}

def job(jid):    return get(JHS + '/' + jid)['job']
def tasklist(jid):return sorted(get(JHS + '/' + jid + '/tasks')['tasks']['task'], key=lambda t: t['id'])
def attempts(jid, tid):
    a = get(f'{JHS}/{jid}/tasks/{tid}/attempts')['taskAttempts'].get('taskAttempt') or []
    return sorted(a, key=lambda x: x['id'])

def attempt_summary(jid):
    rows, ns, nf, nk = [], 0, 0, 0
    for t in tasklist(jid):
        att = attempts(jid, t['id'])
        for a in att:
            if a['state'] == 'SUCCEEDED': ns += 1
            elif a['state'] == 'FAILED':  nf += 1
            elif a['state'] == 'KILLED':  nk += 1
        rows.append([t['id'].split('_')[-1], t['type'], t['state'], len(att),
                     ' '.join(a['id'].rsplit('_', 1)[-1] + '=' + a['state'] for a in att)])
    return rows, ns, nf, nk

jobs = sorted(get(JHS + '?limit=200')['jobs']['job'], key=lambda j: j['id'])
table(['Job', 'Experiment', 'State', 'Expectation'],
      [[j['id'][-4:], LABELS.get(j['id'][-4:], (j['name'], ''))[0], j['state'],
        LABELS.get(j['id'][-4:], ('', ''))[1]] for j in jobs])

---
## 6. Experiment 1 — Normal baseline
**Run:** `bash scripts/run_experiment.sh 01 /results/exp01_baseline NORMAL /data/clean/nasa_access_clean.log`

12 map tasks + 1 reduce, one successful attempt per task. Output = 7 status-code lines.
Reference output every other successful run must match exactly.

In [ ]:
RUN = False  # set True to re-run Exp 1 live (~2.5 min)
if RUN:
    sh('bash ' + PROJ + '/scripts/run_experiment.sh 01 /results/exp01_baseline NORMAL /data/clean/nasa_access_clean.log')

rows, ns, nf, nk = attempt_summary('job_1793346852065_0001')
table(['Task', 'Type', 'TaskState', '#Att', 'Attempts'], rows)
print(f'succeeded={ns}  failed={nf}  killed={nk}')
print('\nBaseline output:')
sh('hdfs dfs -cat /results/exp01_baseline/part-r-*')

---
## 7. Experiment 2 — Malformed record → failure & retries
### 2a — permanent failure (`MALFORMED_RECORD_EXCEPTION`, `mapreduce.map.maxattempts=2`) → **job FAILED**
Every attempt re-reads the split and hits the same `CE06_BAD_RECORD` line →
`attempt_…_m_000000_0 FAILED` → `_1 FAILED` → AM fails the job
(RM diagnostic: *`Task failed task_…_0002_m_000000; Job failed as tasks failed`*).
**Retries cannot fix input-poisoning — the job fails.**

### 2b — transient failure (`FAIL_ONCE`) → **job SUCCEEDED**
Only attempt `_0` throws; retry `_1` skips the marker (`BAD_LINE_SKIPPED_ON_RETRY=12`) and
commits. JobHistory shows `0=FAILED 1=SUCCEEDED` on 11 tasks — **same input, opposite outcome.**

In [ ]:
RUN = False  # Exp 2a live (~2 min)
if RUN:
    sh('bash ' + PROJ + '/scripts/run_experiment.sh 02a /results/exp02a_malformed_fail '
       'MALFORMED_RECORD_EXCEPTION /data/malformed/nasa_access_malformed.log -Dmapreduce.map.maxattempts=2')

for jid, tag in [('job_1793346852065_0002', 'EXP2a  poison + max=2  -> FAILED'),
                 ('job_1793346852065_0003', 'EXP2b  fail-once      -> SUCCEEDED')]:
    rows, ns, nf, nk = attempt_summary(jid)
    print('\n===', tag, '===')
    table(['Task', 'Type', 'TaskState', '#Att', 'Attempts'], rows)
    print(f'succeeded={ns} failed={nf} killed={nk}')

print('\n--- the actual exception, from aggregated YARN logs (job 0002) ---')
p = os.path.join(PROJ, 'evidence/logs/application_1793346852065_0002_failure_stacktraces.log')
with open(p, errors='replace') as f:
    print(''.join(f.readlines()[:14]))

---
## 8. Experiment 3 — `mapreduce.map.maxattempts` (2 vs 4)
Deterministic `ALWAYS_FAIL` (throws in `setup()` on partition 0, same clean input both runs).

| Job | Setting | `m_000000` attempts | Job outcome |
|---|---|---|---|
| `0004` | maxattempts=**2** | `0=FAILED 1=FAILED` (exactly 2) | FAILED |
| `0005` | maxattempts=**4** | `0..1..2..3=FAILED` (exactly 4) | FAILED |

**Reading:** the knob is a per-task attempt budget. For a *deterministic* fault more attempts
only delay the inevitable job failure (but let more sibling tasks finish first: 7 in 3b vs 5 in 3a);
for *probabilistic* faults extra attempts are what keep jobs alive.

In [ ]:
for jid, tag in [('job_1793346852065_0004', 'EXP3a maxattempts=2'),
                 ('job_1793346852065_0005', 'EXP3b maxattempts=4')]:
    rows, ns, nf, nk = attempt_summary(jid)
    t0 = [r for r in rows if r[0] == '000000']
    others = [r for r in rows if r[3] != 1 or r[4].split('=')[-1] != 'SUCCEEDED']
    print(f'=== {tag} ===  job state: {job(jid)["state"]}')
    table(['Task', 'Type', 'TaskState', '#Att', 'Attempts'], t0 + others)
    print(f'tasks fully succeeded before job failed: {sum(1 for r in rows if r[2]=="SUCCEEDED")}/13')
    print()

---
## 9. Experiments 4 & 5 — Speculative execution (the controlled pair)
Identical workload — `SLOW_TASK`, partition 0 sleeps **400 ms per 1000 lines in attempt `_0` only**
(≈296k lines ⇒ ~118 s artificial delay) — only `mapreduce.map.speculative` differs:

| | **Exp 4 — OFF** (`0010`) | **Exp 5 — ON** (`0011`) |
|---|---|---|
| `m_000000` | `0=SUCCEEDED(136,802 ms)` — waited out | `0=KILLED(50,829 ms)` + `1=SUCCEEDED(3,327 ms)` |
| Launched / Killed map tasks | 12 / 0 | **13 / 1** |
| Job | blocked until straggler finished | **69 s** — duplicate rescued it |
| Output | ≡ baseline | ≡ baseline |

**Mechanism:** once sibling maps completed, the AM judged the running attempt slow
(> `slowtaskthreshold` 1.0 × average attempt time), requested a duplicate container,
the duplicate finished first, and RM **KILLED** the original. Only one attempt commits →
speculation trades wasted compute for latency and **never changes results**.

> **Gotcha found:** the preliminary spec-ON run (`0007`, sleep 150 ms) produced **no duplicate** —
> all 3 map slots were continuously busy (a speculative attempt needs a *free* container) and the
> default `speculative-cap-total-tasks=0.01` rounds to ~0 allowed duplicates on a 13-task job.
> The re-run set all caps to 1.0 and lengthened the straggler window. Both runs are kept as evidence.

In [ ]:
RUN = False  # re-run the spec-ON half of the pair live (~1.5 min)
if RUN:
    sh('bash ' + PROJ + '/scripts/run_experiment.sh 05v2 /results/exp05_spec_on SLOW_TASK '
       '/data/clean/nasa_access_clean.log -Dce06.sleep.ms=400 '
       '-Dmapreduce.map.speculative=true -Dmapreduce.reduce.speculative=false '
       '-Dmapreduce.job.speculative.speculativeCap=1.0 '
       '-Dmapreduce.job.speculative.speculative-cap-total-tasks=1.0 '
       '-Dmapreduce.job.speculative.speculative-cap-running-tasks=1.0')

for jid, tag in [('job_1793346852065_0010', 'EXP4v2  spec OFF'),
                 ('job_1793346852065_0011', 'EXP5v2  spec ON '),
                 ('job_1793346852065_0007', 'EXP5 prelim (gotcha: no duplicate)')]:
    rows, ns, nf, nk = attempt_summary(jid)
    t0 = [r for r in rows if r[0] == '000000']
    print(f'=== {tag} === {job(jid)["state"]}')
    table(['Task', 'Type', 'TaskState', '#Att', 'Attempts (id-suffix=state)'], t0)
    c = get(JHS + '/' + jid + '/counters')['jobCounters']['counterGroup']
    jc = next((g for g in c if 'Job Counter' in g['counterGroupName']), None)
    if jc:
        m = {x['name']: x['totalCounterValue'] for x in jc['counter']}
        print('  Launched map tasks =', m.get('LAUNCHED_MAP_TASKS'),
              ' Killed map tasks =', m.get('KILLED_MAP_TASKS'),
              ' Failed map tasks =', m.get('FAILED_MAP_TASKS'))
    print()

print('Output verification (all successful runs vs baseline):')
with open(os.path.join(PROJ, 'evidence/output_verification.txt')) as f:
    print(f.read())

---
## 10. Experiment 6 — Process-level failure: `kill -9` a running attempt
While `attempt_…_0008_m_000000_0` was sleeping, its YarnChild JVM was found via
`pgrep -f …attempt_…_m_000000_0` and killed. NodeManager reported:

```
Container killed on request. Exit code is 137
Container exited with a non-zero exit code 137.
Killed by external signal
```
137 = 128 + 9 = **SIGKILL**. AM transitioned the attempt `RUNNING → FAILED` and launched `_1`,
which succeeded → **job SUCCEEDED**. Contrast with Exp 5: here the loser is **FAILED**
(external kill), there it was **KILLED** (speculative loser).

In [ ]:
rows, ns, nf, nk = attempt_summary('job_1793346852065_0008')
print('=== EXP6  kill -9 ===')
table(['Task', 'Type', 'TaskState', '#Att', 'Attempts'], [r for r in rows if r[3] > 1 or r[0] == '000000'])
print(f'succeeded={ns} failed={nf} killed={nk}')

a = attempts('job_1793346852065_0008', 'task_1793346852065_0008_m_000000')
print('\nDiagnostics of the killed attempt:')
print((a[0].get('diagnostics') or '')[:600])

---
## 11. Experiment 7 — NodeManager failure mid-job (infrastructure level)
**Timeline** (full log: `evidence/logs/exp07_nodemanager_failure_run.log`):
1. **t+0** job RUNNING (~28 %), node `…:38791 RUNNING`.
2. **t+40 s** `yarn --daemon stop nodemanager` → node **SHUTDOWN** immediately (graceful
   deregistration); its containers — **including the ApplicationMaster** — die.
3. **t+40…110 s** app sits **ACCEPTED**; RM diagnostic:
   *`Skipping AM assignment as cluster resource is empty`* (no node to host the AM).
4. **t+110 s** NM restarted → registers as a **new node** (fresh ephemeral port) → RM allocates an
   AM container → **new MRAppMaster recovers job state from HDFS** and re-runs unfinished tasks.
5. **t+188 s** job **SUCCEEDED**, output identical.

**JobHistory signature:** 7 tasks show `0=KILLED … 1000=SUCCEEDED` (RM released them with the
lost node; post-recovery attempts are numbered `_1000`), 5 tasks show only `1000=SUCCEEDED`
(their pre-crash attempt events died with the first AM — history is only as durable as the AM
that writes it). Allowed by `yarn.resourcemanager.am.max-attempts=3` + MR AM recovery.

In [ ]:
RUN = False  # full Exp 7 live (~4 min; stops/starts the NodeManager safely)
if RUN:
    sh('bash ' + PROJ + '/scripts/run_nodemanager_failure.sh')

rows, ns, nf, nk = attempt_summary('job_1793346852065_0009')
print('=== EXP7  NodeManager failure ===')
table(['Task', 'Type', 'TaskState', '#Att', 'Attempts'], rows)
print(f'succeeded={ns} failed={nf} killed={nk}   (job: {job("job_1793346852065_0009")["state"]})')

app = get(RM + '/ws/v1/cluster/apps/application_1793346852065_0009')['app']
print('\nRM final state:', app['state'], app['finalStatus'])

print('\n--- current nodes (old NM IDs remain SHUTDOWN) ---')
nodes = get(RM + '/ws/v1/cluster/nodes')['nodes']['node']
table(['NodeID', 'State'], [[n['id'], n['state']] for n in nodes])

---
## 12. Results overview — all 11 jobs
| Job | Experiment | Outcome | Key evidence |
|---|---|---|---|
| 0001 | 1 Baseline | SUCCEEDED | 13 tasks × 1 SUCCEEDED |
| 0002 | 2a poison, max=2 | **FAILED** | `_0,_1=FAILED`, RM: `Job failed as tasks failed` |
| 0003 | 2b fail-once | SUCCEEDED | 11 tasks `0=FAILED 1=SUCCEEDED`, output ≡ baseline |
| 0004 | 3a max=2 | **FAILED** | `m_000000`: exactly 2 attempts |
| 0005 | 3b max=4 | **FAILED** | `m_000000`: exactly 4 attempts |
| 0010 | 4 spec OFF | SUCCEEDED | straggler `0=SUCCEEDED(136.8 s)` alone |
| 0011 | 5 spec ON | SUCCEEDED | `0=KILLED(50.8 s) 1=SUCCEEDED(3.3 s)`, 69 s job |
| 0008 | 6 kill -9 | SUCCEEDED | exit code 137, `_1` retry won |
| 0009 | 7 NM failure | SUCCEEDED | 7 × `0=KILLED 1000=SUCCEEDED`, AM recovered |

In [ ]:
# Full summary: attempts breakdown per job + saved console RESULT lines
rows = []
for j in jobs:
    _, ns, nf, nk = attempt_summary(j['id'])
    rows.append([j['id'][-4:], LABELS.get(j['id'][-4:], (j['name'], ''))[0][:44],
                 j['state'], ns + nf + nk, ns, nf, nk])
table(['Job', 'Experiment', 'State', 'Att', 'Succ', 'Fail', 'Killed'], rows)

print('\n--- RESULT lines from console logs ---')
logdir = os.path.join(PROJ, 'evidence/logs')
for fn in sorted(os.listdir(logdir)):
    if fn.endswith('_run.log') or fn == 'final_run.log':
        with open(os.path.join(logdir, fn), errors='replace') as f:
            for line in f:
                if line.startswith('RESULT='):
                    print(f'{fn:45s} {line.strip()}')

In [ ]:
# Counter highlights (retry volume + speculation proof)
def counter_map(jid):
    gs = get(JHS + '/' + jid + '/counters')['jobCounters']['counterGroup']
    out = {}
    for g in gs:
        for x in g['counter']:
            out[x['name']] = x['totalCounterValue']
    return out

for jid, tag in [('job_1793346852065_0003', 'EXP2b retries'),
                 ('job_1793346852065_0010', 'EXP4v2 spec OFF'),
                 ('job_1793346852065_0011', 'EXP5v2 spec ON')]:
    m = counter_map(jid)
    keys = ['LAUNCHED_MAP_TASKS', 'KILLED_MAP_TASKS', 'FAILED_MAP_TASKS',
            'MAP_INPUT_RECORDS', 'MAP_OUTPUT_RECORDS', 'REDUCE_OUTPUT_RECORDS']
    print(tag, {k: m.get(k) for k in keys if m.get(k) is not None})

print('\nCE06 custom counters of the FAIL_ONCE job (client-computed, from its log):')
with open(os.path.join(PROJ, 'evidence/logs/exp02b_fail_once_run.log'), errors='replace') as f:
    for line in f:
        if line.startswith('CE06 COUNTERS'):
            print(line.strip())

## 13. What a single-node laptop honestly CANNOT show
- **Node LOST via heartbeat expiry** — a graceful stop deregisters instantly (SHUTDOWN); a hard
  crash of the only NM also kills the only AM executor, so the demo degenerates into AM relaunch
  rather than retry-on-surviving-node.
- **Rack-aware placement / network partitions** — needs ≥ 2 racks.
- **HDFS re-replication after DataNode loss** — with `dfs.replication=1` and one DN there is no
  second copy (documented, deliberately not staged).
- **Speculation on a busy multi-node cluster** — with 3 map slots a duplicate needs a genuinely
  free slot; we engineered the window (see the gotcha in §9) instead of relying on load.
- Timings reflect one shared kernel — not datacenter tail latency.

All of the above are reported as environment limits in `FINAL_REPORT.md` §7 rather than faked.

## 14. Conclusion & learning outcomes
1. **Failures & retries:** observed YARN's automatic per-task retries — *transient* failure
   recovered (`FAIL_ONCE`), *permanent* failure exhausted attempts and failed the job.
2. **`maxattempts` semantics:** proven attempt-for-attempt (exactly 2 vs exactly 4) and its
   effect on the job verdict.
3. **Speculative execution:** duplicate attempt launched, straggler KILLED at 50.8 s, job 69 s vs
   137 s unaided — with byte-identical output (correctness by commit election).
4. **Out-of-band & infrastructure failures:** SIGKILL (exit 137) recovered by retry; NodeManager
   loss survived via AM relaunch + HDFS job recovery.
5. **Evidence discipline:** every claim above is backed by JobHistory REST JSON, RM diagnostics,
   aggregated logs, UI snapshots and output diffs (106 files under `evidence/`).

**Deliverables:** `FINAL_REPORT.md` · `DEMONSTRATION_GUIDE.md` · `VIVA_QA.md` · `README.md` ·
`readme_for_ppt.txt` (presentation kit) · `evidence/` · `config/`.

In [ ]:
# Evidence inventory — everything the report cites
root = os.path.join(PROJ, 'evidence')
count = 0
for dirpath, _, files in os.walk(root):
    count += len(files)
print(f'{count} evidence files')
for sub in sorted(os.listdir(root)):
    p = os.path.join(root, sub)
    if os.path.isdir(p):
        print(f'  evidence/{sub}/  ({len(os.listdir(p))} files)')
    else:
        print(f'  evidence/{sub}')

print('\n--- one-page results table (head) ---')
with open(os.path.join(root, 'REPORT_EVIDENCE_SUMMARY.txt'), errors='replace') as f:
    lines = f.readlines()
print(''.join(lines[:26]))